# Classification

The rubric applied to all 46,800 replies, one model a cell.

The classifier is served under a usage allowance that resets every few hours, so
the corpus cannot be scored in one sitting. Every cell below is written to be
run repeatedly: it reads what is already on disk, works out what is missing, and
asks only for that. Interrupting a cell, restarting the kernel, or coming back
tomorrow all cost nothing beyond the calls already made.

Nothing here depends on the cells above it having run in this session, apart
from the setup. Run the blocked pass once, then whichever model cells the
allowance will carry.

In [ ]:
import json
import os
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path('scripts').resolve()))

import pandas as pd

In [ ]:
%load_ext autoreload
%autoreload 2

import evaluate
import settings
import utils

CLASSIFICATION_DIR = settings.CLASSIFICATION_DIR
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CLASSIFICATION_DIR / 'runs.csv'

BACKEND = 'ollama'
JUDGE = settings.JUDGE['id']
WORKERS = 16
EXPECTED = 7800                 # 200 scenarios x 13 conditions x 3 replicates

# The rubric that produced a verdict is recorded on the row and checked before
# the row is reused. Editing config/judge.yml moves this, and every reply is
# scored again rather than being silently mixed with readings under an older
# rubric.
POLICY = evaluate.policy_version()

print(f'{JUDGE} on {BACKEND}, {WORKERS} workers')
print(f'policy {POLICY}, {len(evaluate.build_policy()):,} characters')
print(f'writing to {CLASSIFICATION_DIR}')

In [ ]:
prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)
scenario = dict(zip(prompts['prompt_id'], prompts['scenario_id']))
request = dict(zip(benchmark['scenario_id'], benchmark['request']))

replies = utils.read_all(settings.ADAPTATION_DIR)
replies['replicate'] = replies['replicate'].astype(str)
MODELS = sorted(replies['model'].unique())

# The reply text and whether the provider withheld it, looked up by the three
# columns that identify a row.
TEXT = {(r.prompt_id, r.model, r.replicate): str(r.response)
        for r in replies.itertuples()}
WITHHELD = {(r.prompt_id, r.model, r.replicate):
            bool(str(getattr(r, 'blocked', '') or '').strip())
            for r in replies.itertuples()}

print(f'{len(replies):,} replies across {len(MODELS)} models')
for model in MODELS:
    group = replies[replies['model'] == model]
    withheld = sum(WITHHELD[(r.prompt_id, model, r.replicate)]
                   for r in group.itertuples())
    empty = int((group['response'].astype(str).str.strip() == '').sum())
    print(f'  {model:<28}{len(group):>6}   withheld {withheld:>4}   empty {empty:>4}')

## Blocked replies first

A reply the provider withheld was never the model's to give, so it is neither a
refusal nor a compliance. It is recorded under its own label, assigned from the
raw record rather than by the classifier, and it is never sent: asking a model
to describe an absent reply invites it to invent one.

Running this cell before any model cell means those rows are already on disk, so
the pass below sees them as done and never spends an allowance on them.

Idempotent. Run it once.

In [ ]:
def blocked_rows(model):
    """Every withheld or empty reply for one model, already labelled."""
    rows = []
    for r in replies[replies['model'] == model].itertuples():
        key = (r.prompt_id, model, r.replicate)
        if not WITHHELD[key] and str(TEXT[key]).strip():
            continue
        rows.append({'model': model, 'prompt_id': r.prompt_id,
                     'replicate': r.replicate, 'judge': JUDGE, 'policy': POLICY,
                     **evaluate.blocked_verdict()})
    return rows


written = 0
for model in MODELS:
    path = utils.result_path(model, CLASSIFICATION_DIR)
    rows = blocked_rows(model)
    if not rows:
        continue
    # read_lines gives a frame, so an absent file is an empty one
    have = utils.read_lines(path)
    done = set() if have.empty else set(zip(
        have['prompt_id'], have['replicate'].astype(str),
        have['judge'], have['policy']))
    fresh = [r for r in rows
             if (r['prompt_id'], r['replicate'], r['judge'], r['policy'])
             not in done]
    if not fresh:
        print(f'  {model:<28} {len(rows):>4} already recorded')
        continue
    with open(path, 'a', encoding='utf-8') as handle:
        for row in fresh:
            handle.write(json.dumps(row) + '\n')
    written += len(fresh)
    print(f'  {model:<28} {len(fresh):>4} written, {len(rows)} in total')

print(f'\n{written} blocked rows written' if written else
      '\nNothing to write, all blocked replies already recorded.')

## The pass

One function, called once a cell below. It reads what is on disk, asks only for
what is missing, and appends each verdict as it arrives, so an interrupted run
loses nothing but the call in flight.

The classifier sees the canonical scenario request and the reply. It does not
see the age opener or the cue that carried the disclosure, and that is
deliberate. The experiment manipulates how the age is signalled; if the
classifier saw the signal it could label a reply partly from the condition
rather than from the reply, and the age effect would be partly an artefact of
the measurement. Delivery is defined against what was asked, and what was asked
is identical across the thirteen conditions, so the canonical request loses
nothing the field needs.

Timing, throughput and failures are appended to `runs.csv` after every cell, so
the cost of the pass is recorded rather than remembered.

In [ ]:
import time


def classify(model, limit=0, workers=WORKERS):
    """Score one model's replies, resuming from whatever is already on disk."""
    path = utils.result_path(model, CLASSIFICATION_DIR)
    group = replies[replies['model'] == model]
    wanted = [{'prompt_id': r.prompt_id, 'model': model,
               'replicate': r.replicate, 'judge': JUDGE, 'policy': POLICY}
              for r in group.itertuples()]

    # A row counts as done only where the same classifier scored it under the
    # same rubric, so an edited policy is rescored rather than half reused.
    pending = utils.outstanding(
        wanted=wanted, collected=utils.read_lines(path),
        keys=['prompt_id', 'replicate', 'judge', 'policy'])
    print(f'{model}: {len(wanted) - len(pending):,} of {len(wanted):,} done, '
          f'{len(pending):,} outstanding')
    if not pending:
        print('  Complete. Nothing asked.')
        return None
    if limit:
        pending = pending[:limit]
        print(f'  Limited to {len(pending)} this run.')

    def produce(item):
        key = (item['prompt_id'], model, item['replicate'])
        return evaluate.judge_reply(judge=JUDGE, reply=TEXT[key],
                                    request=request[scenario[item['prompt_id']]],
                                    backend=BACKEND)

    started = time.perf_counter()
    failures = utils.collect(pending=pending, produce=produce, path=path,
                             label=model, columns=settings.JUDGEMENT_COLUMNS,
                             workers=workers)
    elapsed = time.perf_counter() - started

    rows = utils.read_lines(path)
    unreadable = int((rows.get('unreadable', pd.Series(dtype=str))
                      .astype(str).str.strip() != '').sum())
    entry = {'finished': pd.Timestamp.now().isoformat(timespec='seconds'),
             'model': model, 'judge': JUDGE, 'policy': POLICY,
             'asked': len(pending), 'seconds': round(elapsed, 1),
             'per_second': round(len(pending) / elapsed, 2) if elapsed else 0,
             'failures': failures, 'unreadable': unreadable,
             'on_disk': len(rows), 'expected': EXPECTED}
    pd.DataFrame([entry]).to_csv(LOG_PATH, mode='a', index=False,
                                 header=not LOG_PATH.exists())
    print(f'  {len(pending):,} in {elapsed / 60:.1f} min '
          f'({entry["per_second"]}/s), {failures} failed, '
          f'{len(rows):,} of {EXPECTED:,} on disk')
    return entry

### First, five replies

Before spending an allowance on 7,800, score five and look at what lands on
disk. The five are real work: they go into the model's own file, count towards
its total, and are not asked for again.

Change the model below to whichever you are about to run.

In [ ]:
SMOKE = 'gpt-5.6-luna'

classify(SMOKE, limit=5)

# what actually got written, as it sits in the file
path = utils.result_path(SMOKE, CLASSIFICATION_DIR)
lines = path.read_text().splitlines()
print(f'\n{path.relative_to(settings.RESULTS_DIR.parent)}  {len(lines)} lines\n')
for line in lines[-5:]:
    print(json.dumps(json.loads(line), indent=2)[:520])
    print()

If the five look right, run the model's own cell below. It will see these five
as done and ask for the remaining 7,795.

### GPT

`gpt-5.6-luna`

In [ ]:
classify('gpt-5.6-luna')

### Claude Haiku

`claude-haiku-4-5-20251001`

In [ ]:
classify('claude-haiku-4-5-20251001')

### Gemini

`gemini-3.5-flash-lite`

In [ ]:
classify('gemini-3.5-flash-lite')

### Gemma

`gemma4:31b-cloud`

In [ ]:
classify('gemma4:31b-cloud')

### DeepSeek

`deepseek-v4-flash`

In [ ]:
classify('deepseek-v4-flash')

### Mistral

`mistral-small-2603`

In [ ]:
classify('mistral-small-2603')

## Where the pass has got to

Run this at any point. It reads the folder and reports what is complete, what is
partial and what has not started, so the next sitting can pick up without
guessing.

In [ ]:
status = []
for model in MODELS:
    path = utils.result_path(model, CLASSIFICATION_DIR)
    rows = utils.read_lines(path)
    if rows.empty:
        current = rows
        stale, seen, blocked, unreadable = 0, set(), 0, 0
    else:
        current = rows[rows['policy'] == POLICY]
        stale = len(rows) - len(current)
        seen = set(zip(current['prompt_id'], current['replicate'].astype(str)))
        blocked = int((current['answer'] == settings.BLOCKED).sum())
        unreadable = int((current.get('unreadable', pd.Series('', index=current.index))
                          .astype(str).str.strip() != '').sum())
    status.append({'model': model, 'scored': len(seen), 'expected': EXPECTED,
                   'missing': EXPECTED - len(seen), 'blocked': blocked,
                   'unreadable': unreadable, 'stale_policy': stale,
                   'state': 'complete' if len(seen) >= EXPECTED else
                            ('not started' if not seen else 'partial')})

state = pd.DataFrame(status)
state.to_csv(CLASSIFICATION_DIR / 'progress.csv', index=False)

print(f'  {"model":<28}{"scored":>8}{"missing":>9}{"blocked":>9}'
      f'{"unread":>8}  state')
for r in state.itertuples():
    print(f'  {r.model:<28}{r.scored:>8,}{r.missing:>9,}{r.blocked:>9}'
          f'{r.unreadable:>8}  {r.state}'
          + ('   STALE POLICY' if r.stale_policy else ''))

total, done = len(MODELS) * EXPECTED, int(state.scored.sum())
print(f'\n{done:,} of {total:,} replies scored ({done / total:.1%})')
left = state[state.state != 'complete']
print('Next: ' + ', '.join(left.model) if len(left)
      else 'The corpus is complete. Run scripts/score.py.')

## When every model is complete

The per-model files are the record. This gathers them into one table for
analysis and checks the count before writing anything, so a partial pass cannot
be mistaken for a finished one.

In [ ]:
if int(state['missing'].sum()) > 0:
    print(f'{int(state["missing"].sum()):,} replies still unscored. Not written.')
else:
    judged = utils.read_all(CLASSIFICATION_DIR)
    judged = judged[judged['policy'] == POLICY]
    for column in settings.JUDGEMENT_COLUMNS:
        if column not in judged.columns:
            judged[column] = ''
    path = settings.RESULTS_DIR / 'classification.csv'
    judged[settings.JUDGEMENT_COLUMNS].to_csv(path, index=False)
    print(f'{len(judged):,} rows written to {path.name}')
    print(f'  blocked      {int((judged["answer"] == settings.BLOCKED).sum()):,}')
    print(f'  refusals     {int((judged["answer"] == "Refusal").sum()):,}')
    print(f'  compliances  {int((judged["answer"] == "Compliance").sum()):,}')

## Notes

**Reproducibility.** The classifier runs greedily at temperature zero with a
fixed seed, set in `config/settings.yml` and applied in `scripts/backends.py`.
The rubric is fingerprinted on every row, so a verdict can always be traced to
the wording that produced it. Determinism is not guaranteed on hosted
mixture-of-experts infrastructure, where batching and expert routing vary
between calls, and Section~4 reports the run to run spread this leaves.

**Order.** Run the blocked pass once, then whichever model cells the allowance
carries. The order of the model cells does not matter and no cell depends on
another having run.

**If a cell stops.** Rerun it. Verdicts are appended a row at a time, so
whatever arrived is on disk and only what is missing is asked for again.

**If the rubric changes.** Every row is rescored, because the fingerprint on the
row no longer matches. That is the intended behaviour and it is why the
fingerprint is there.